In [74]:
import pandas as pd
import networkx as nx
import random
from collections import defaultdict

In [229]:
#  Import Data
relationships = pd.read_csv("data/hero_relationships.csv", keep_default_na=False).drop(columns=["Unnamed: 7", "Global Notes"])
hero_tier_list = pd.read_csv("data/hero_tier_list.csv").drop(columns=["Unnamed: 3"])


# Build lookup data
all_hero_names = relationships["name"].to_list()
# Numeric map of tiers 
tier_map = {
    "S": 5,
    "A": 4,
    "B": 3,
    "C": 2,
    "D": 1
}
tier_map_rev = {
    score: tier
    for tier, score in tier_map.items()
}

# Add a tier score to the tier list for scoring
hero_tier_list["tier_score"] = hero_tier_list["Tier"].map(tier_map)


In [ ]:
# Global Helper Functions
def get_hero_best_positions(hero_name: str | list[str], tier=1, verbose=False) -> dict[str, list[dict]]:
    output = {}

    names = [hero_name] if isinstance(hero_name, str) else hero_name.copy()

    for name in names:
        hero_rows = hero_tier_list[hero_tier_list["Name"] == name]
        max_score = hero_rows["tier_score"].max()
        target_score = max_score - (tier - 1)

        best_positions = hero_rows.loc[
            hero_rows["tier_score"] == target_score,
            ["Position", "Tier"]
        ].to_dict("records")

        if not best_positions:
            print(f"No heroes found under the name {name}.")
            continue

        position_names = [position["Position"] for position in best_positions]
        print_out = f"{name}'s best position is {', '.join(position_names)}."

        if verbose:
            print(print_out)

        output[name] = best_positions

    return output

# Get the best heroes for a position
def get_position_best_heroes(position: str | list[str], tier=1, verbose=False) -> dict[str, list[dict]]:
    output = {}

    positions = [position] if isinstance(position, str) else position.copy()

    for position in positions:
        position_rows = hero_tier_list[hero_tier_list["Position"] == position]
        max_score = position_rows["tier_score"].max()
        target_score = max_score - (tier - 1)

        best_heroes = position_rows.loc[
            position_rows["tier_score"] == target_score,
            ["Name", "Tier"]
        ].to_dict("records")

        if not best_heroes:
            print(f"No heroes found for the {position} position.")
            continue

        hero_names = [hero["Name"] for hero in best_heroes]
        print_out = f"Best heroes for the {position.lower()} position are {', '.join(hero_names)}."

        if verbose:
            print(print_out)

        output[position] = best_heroes

    return output

best_positions = get_hero_best_positions("Bart", verbose=True, tier=1)
best_heroes = get_position_best_heroes(["Top", "Mid"], verbose=True)

Bart's best position is Top.
Best heroes for the top position are Frank, Kid, Wukong, Foso.
Best heroes for the mid position are Aurelio, Merisi, Wolfgang, Foso.


In [81]:
# Confirm heroes in relationships are == to heroes in tiers 
heroes_t = set(hero_tier_list["Name"].unique().tolist())
all_hero_names = set(all_hero_names)

mismatches = []
for hero in heroes_t:
    if hero in all_hero_names:
        continue
    mismatches.append((hero))

if (len(heroes_t) != len(all_hero_names)) or mismatches:
    print("Names do not match.")
else:
    print("Names match.")

Names match.


In [ ]:
# Build tag -> hero lookup so we can pump in heros if we see tags for synergies etc.
lookup = defaultdict(list)
for hero in all_hero_names:
    hero_data = relationships[relationships["name"] == hero]
    hero_tags = hero_data["tags"].tolist()[0].split(",")
    for tag in hero_tags:
        lookup[tag].append(hero)
print(lookup)

defaultdict(<class 'list'>, {'Tank': ['Big Foot', 'Bart', 'Fatty White', 'Frank', 'Merisi', 'Matata', 'Gillis', 'Peter', 'Miki', 'Cubey'], 'Debuff': ['Big Foot', 'Lady Deadfire', 'Crank', 'Shougong Lei', 'Tiger Boy', 'Felicity'], 'Lane Bully': ['Qube', 'Merisi', 'Felyn', 'Wolfgang'], 'High Damage': ['Qube'], 'Crit Immunity': ['Bart'], 'Thorn': ['Bart'], 'Crit': ['Bariel', 'Raven', 'Quinn', 'Manta', 'BaJie'], 'One Slap Chap': ['Chen Fengcheng', 'Reinhardt'], 'Armour': ['Fatty White'], 'Dispell': ['Fatty White'], 'Search': ['Blocker', 'Beverly'], 'ADC': ['Elemi', 'Deep Space', 'Omaha', 'Quinn', 'Ada', 'Bond'], 'Multi-Attacker': ['Elemi', 'Kamaitachi', 'Mo', 'Zealot'], 'Lifesteal': ['Fernan', 'Nihil'], 'True Damage': ['Digo', 'Mihawk', 'Manta'], 'Targeting': ['Xiangxi Ke', 'Leon', 'Lubos'], 'Assassin': ['Xiangxi Ke', 'Lan', 'Gillis', 'Raven', 'Manta', 'Lubos'], 'Kill Dependent': ['Lan'], 'Burst': ['Lan'], 'Skirmisher': ['Lan', 'Aurelio', 'Raven', 'Ada', 'Bond', 'Babe', 'Anna', 'Zealot'], 

Masteries

In [258]:
# Setup mastery functions.  This does not live on the graph
def create_empty_masteries(G):
    return {
        hero: {
            position: 0
            for position in G.nodes[hero]["tiers"]
        }
        for hero in G.nodes
    }

def set_mastery(masteries, hero: str, position: str, level: int):
    if hero not in masteries:
        raise ValueError(f"Unknown hero: {hero}")

    if position not in masteries[hero]:
        raise ValueError(f"{hero} cannot play {position}")

    if not 0 <= level <= 7:
        raise ValueError("Mastery must be between 0 and 7")

    masteries[hero][position] = level


# Build hero tier and master lookups
tiers = defaultdict(dict)
t1_masteries = create_empty_masteries(G_master)
t2_masteries = create_empty_masteries(G_master)

# Build t1 masteries  - for every position that this hero can be in, what is the mastery of the actual players?
for _, row in hero_tier_list.iterrows():
    tiers[row["Name"]][row["Position"]] = row["tier_score"]

    set_mastery(t1_masteries, row["Name"], row["Position"], random.randint(0, 7))

print(t1_masteries["Frank"])
print(tiers["Frank"])

# Build t2 masteries
for _, row in hero_tier_list.iterrows():
    tiers[row["Name"]][row["Position"]] = row["tier_score"]
    set_mastery(t2_masteries, row["Name"], row["Position"], random.randint(0, 7))
print(t1_masteries["Frank"])
print(tiers["Frank"])


{'Mid': 5, 'Top': 1}
{'Mid': 4, 'Top': 5}
{'Mid': 5, 'Top': 1}
{'Mid': 4, 'Top': 5}


Master Graph

In [259]:
# Make the graph
G_master = nx.MultiDiGraph()

# Make a node for every hero
G_master.add_nodes_from(all_hero_names)

# Add the position tier data
nx.set_node_attributes(G_master, tiers, name="tiers")
nx.set_node_attributes(G_master, t1_masteries, name="t1_masteries")
nx.set_node_attributes(G_master, t2_masteries, name="t2_masteries")

#G_master.nodes["Aurelio"]["masteries"]["Jungler"] += 1  # TODO build a helpder to rank up a mastery on a hero's postion
G_master.nodes["Zealot"]

{'tiers': {'Jungler': 5, 'Top': 3},
 't1_masteries': {'Jungler': 1, 'Top': 6},
 't2_masteries': {'Jungler': 3, 'Top': 5}}

In [171]:
# Resolve tags to heros
def resolve_tags(raw: str, lookup: dict, all_heroes: set) -> list[str]:
    if not raw or raw.strip().lower() == "none":
        return []
    targets = []
    for token in raw.split(","):
        token = token.strip()
        if token in all_heroes:
            targets.append(token)
        elif token in lookup:
            targets.extend(lookup[token])  # tag -> heroes
    return list(set(targets))  # deduplicate

In [ ]:
# Edge columns
edge_types = {
    "synergies": "synergy",
    "counters": "counter",
    "countered_by": "countered_by",
    "anti-synergy": "anti_synergy",
}

for _, row in relationships.iterrows():
    hero = row["name"]
    for col, edge_type in edge_types.items():
        targets = resolve_tags(str(row[col]), lookup, all_hero_names)
        for target in targets:
            if target == hero:
                continue
            G_master.add_edge(hero, target, type=edge_type)

OutEdgeDataView([('Kid', 'Lan', {'type': 'synergy'}), ('Kid', 'Xiangxi Ke', {'type': 'synergy'}), ('Kid', 'Gillis', {'type': 'countered_by'}), ('Kid', 'Mo', {'type': 'synergy'}), ('Kid', 'Lubos', {'type': 'synergy'}), ('Kid', 'Manta', {'type': 'synergy'}), ('Kid', 'Raven', {'type': 'synergy'}), ('Kid', 'Big Foot', {'type': 'countered_by'}), ('Kid', 'Peter', {'type': 'countered_by'}), ('Kid', 'Qube', {'type': 'countered_by'}), ('Kid', 'Bart', {'type': 'countered_by'}), ('Kid', 'Miki', {'type': 'countered_by'}), ('Kid', 'Hass', {'type': 'countered_by'}), ('Kid', 'Cubey', {'type': 'countered_by'}), ('Kid', 'Frank', {'type': 'countered_by'}), ('Kid', 'Merisi', {'type': 'countered_by'}), ('Kid', 'Fatty White', {'type': 'countered_by'}), ('Kid', 'Matata', {'type': 'countered_by'})])

Start Draft

In [ ]:
# If mastery for hero + position is 0, then that player cannot play that hero at all
def build_position_availability(masteries):
    available = defaultdict(set)

    for hero, position_masteries in masteries.items():
        for position, mastery in position_masteries.items():
            if mastery > 0:
                available[position].add(hero)

    return available

# Build out a subgraph
def build_match_graph(G, team_a_available, team_b_available):
    all_available = set().union(*team_a_available.values(), *team_b_available.values())
    return G.subgraph(all_available).copy()

In [ ]:
# Build position availabilities
t1_available = build_position_availability(masteries)
t2_available = build_position_availability(masteries) # This will have to be manually done for team 2

# Who has already been picked?
t1_picked = {} 
t2_picked = {}

# Who has already been banned?
t1_banned = set()
t2_banned = set()

# Build out the match graph
G_match = build_match_graph(G_master, t1_available, t2_available)

Score a Hero

In [ ]:
def flatten_available(available: dict) -> set:
    return set().union(*available.values())

def score_hero(
        G, 
        candidate,
        position, # for tier and mastery scoring
        t1_available: set, t1_picked: set, 
        t2_available: set, t2_picked: set
    ):
    
    # Team Comp Weights
    w_countered_picked = 3
    w_countered_available = 1.5
   
    w_synergy_picked = 1
    w_synergy_available = 0.5
    
    w_counter_picked = 2
    w_counter_available = 1
    
    w_a_synergy_picked = 0.5
    w_a_synergy_available = 0.25

    # Mastery weight
    w_mastery = 1

    # Tier Weight
    w_tier = 2

    # Init score
    score = 0
    explanation = defaultdict(set)

    t1_available_distinct = flatten_available(t1_available)
    t2_available_distinct = flatten_available(t2_available)

    #1. Is there anything that counters this hero that can be picked against it?  
    # TODO for each hero that counteres this one, how good/strong is that counter?  Score reductions should be based on how strong the counter is
    # TODO for each hero that counter's this one, has been picked already?  Heroes that have been picked should carry a stronger importance
    countered_by = {v for _, v, d in G.out_edges(candidate, data=True) if d["type"] == "countered_by"}
    for hero in countered_by:
        if hero in t2_picked:
            score -= w_countered_picked
            explanation["countered_by"].add(hero)
        elif hero in t2_available_distinct:
            score -= w_countered_available
            explanation["countered_by_possible"].add(hero)

    # 2. What synergies are available for this hero
    synergies = {v for _, v, d in G.out_edges(candidate, data=True) if d["type"] == "synergy"}
    for hero in synergies:
        if hero in t1_picked:
            score += w_synergy_picked
            explanation["synergy"].add(hero)
        elif hero in t1_available_distinct:
            score += w_synergy_available
            explanation["synergy_possible"].add(hero)

    # 3. Are there opportunities to counter enemy heroes?
    # TODO heros that we can counter with higher masteries should carry a higher weight over ones that aren't as strong
    counters = {v for _, v, d in G.out_edges(candidate, data=True) if d["type"] == "counter"}
    for hero in counters:
        if hero in t2_picked:
            score += w_counter_picked
            explanation["counters"].add(hero)
        elif hero in t2_available_distinct:
            score += w_counter_available
            explanation["counters_possible"].add(hero)

    # 4. Are there any issues with picking this hero with our current heroes?
    anti_synergy = {v for _, v, d in G.out_edges(candidate, data=True) if d["type"] == "anti_synergy"}
    for hero in anti_synergy:
        if hero in t1_picked:
            score -= w_a_synergy_picked
            explanation["a_synergy"].add(hero)
        elif hero in t1_available_distinct:
            score -= w_a_synergy_available
            explanation["a_synergy_possible"].add(hero)

    # 5 - Add weight for tier
    score += G.nodes[candidate]["tiers"][position] * w_tier
    explanation["position_tier"].add(tier_map_rev[G.nodes[candidate]["tiers"][position]])

    # 6 - Add weight for mastery
    score += G.nodes[candidate]["masteries"][position] * w_mastery
    explanation["position_mastery"].add(G.nodes[candidate]["masteries"][position])

    return score, explanation

In [ ]:
def recommend_pick(G, lane, t1_available, t1_picked, t2_available, t2_picked):
    
    # Score all candidates for the requested lane
    candidates = t1_available[lane]
    results = {
        hero: score_hero(G, hero, lane, t1_available, t1_picked, t2_available, t2_picked)
        for hero in candidates
    }
    
    # Get the best pick for this lane, Each value is: (score, explanation)
    best = max(results, key=lambda hero: results[hero][0])
    best_score, best_explanation = results[best]
    
    # Check if best pick scores higher in another lane
    other_results = {
        other_lane: score_hero(G, best, other_lane, t1_available, t1_picked, t2_available, t2_picked)
        for other_lane, pool in t1_available.items()
        if other_lane != lane and best in pool
    }

    flag = None

    if other_results :
        best_alt_lane = max(
            other_results,
            key=lambda other_lane: other_results[other_lane][0],
        )
        alt_score, _ = other_results[best_alt_lane]

        if alt_score > best_score:
            flag = (
                f"If you can, consider {best} for {best_alt_lane} instead "
                f"(scores {alt_score:.2f} in {best_alt_lane} vs {best_score:.2f} in {lane})"
            )
    
    return best, best_score, best_explanation, flag, results

# Recommend a Pick
best, score, explanation, flag, all_results = recommend_pick(G_match, "Top", t1_available, t1_picked, t2_available, t2_picked)
print(f"Recommended: {best} ({score})")
for reason in explanation:
    output = f"- {reason}:"
    for hero in explanation[reason]:
        output += f" {hero},"
    print(output)
if flag:
    print(f"WARNING: {flag}")
print("")

# Recommend what the enemy should pick (we could run this to suggest bans)
best, score, explanation, flag, all_scores = recommend_pick(G_match, "Mid", t2_available, t2_picked, t1_available, t1_picked)
print(f"Recommended: {best} ({score})")
for reason in explanation:
    output = f"- {reason}:"
    for hero in explanation[reason]:
        output += f" {hero},"
    print(output)
if flag:
    print(f"WARNING: {flag}")


# Recommend a Ban
#best, score, flag, all_scores = recommend_ban(G_match, "Support", t1_available, t1_picked, t2_available, t2_picked)
#print(f"Recommended: {best} ({score})")
#print(f"All scores: {sorted(all_scores.items(), key=lambda x: x[1], reverse=True)}")
#if flag:
#    print(f"WARNING: {flag}")'''

Recommended: Foso (29.0)
- countered_by_possible: Hass,
- synergy_possible: Miki,
- synergy: Palulu,
- counters: Lan,
- counters_possible: Qin Hu, Kaka, Babe, Bond, Anna, Omaha, Ada, Zealot, Raven, Aurelio,
- position_tier: S,
- position_mastery: 7,

Recommended: Digo (22.0)
- synergy_possible: Big Foot, Peter, Bart, Miki, Cubey, Frank, Merisi, Matata,
- counters_possible: Xiangxi Ke, Gillis, Fatty White, Manta, Raven,
- position_tier: B,
- position_mastery: 7,


Draft Functions

In [ ]:
# Hero set manipulation functions
def pick_hero(
        G, t_picked: dict[str, set[str]], hero: str,
        t1_available: dict[str, set[str]],
        t2_available: dict[str, set[str]]
    ):

    # This hero is no longer available
    for pool in t1_available.values():
        pool.discard(hero)
    for pool in t2_available.values():
        pool.discard(hero)
    
    # Set positions this new hero could play
    t_picked[hero] = {
        position
        for position, mastery in G.nodes[hero]["masteries"].items()
        if mastery > 0
    }

    changed = True
    while changed:
        changed = False
        
        # See what positions are locked
        locked_positions = {
            next(iter(positions))
            for positions in t_picked.values()
            if len(positions) == 1
        }

        for possible_positions in t_picked.values():
            if len(possible_positions) == 1:
                continue

            previous_positions = possible_positions.copy()
            possible_positions.difference_update(locked_positions)

            if possible_positions != previous_positions:
                changed = True

            if not possible_positions:
                raise ValueError(f"{hero} has no valid remaining positions")

    return t_picked

def ban_hero(t_banned, hero, t1_available, t2_available):
    for pool in t1_available.values():
        pool.discard(hero)
    for pool in t2_available.values():
        pool.discard(hero)
    t_banned.add(hero)
    return t_banned

def see_current_draft(t_picked: set):
    return t_picked

def see_current_banned(t_banned: set):
    return t_banned

In [ ]:
ban_hero(t1_banned, "Tivie")

{'Tivie'}

In [ ]:
pick_hero(G_match, t1_picked, "Palulu", t1_available, t2_available)

{'Palulu': {'Mid', 'Support'}}

In [241]:
pick_hero(G_match, t2_picked, "Lan")

{'Lan': {'Mid'}}

In [224]:
# See current draft
see_current_draft(t1_picked)

{'Gang': {'Bot'},
 'Anna': {'Support'},
 'Big Foot': {'Top'},
 'Digo': {'Mid'},
 'Leon': {'Jungler'}}